# Imports 

### Import Libraries

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path[:3])

['c:\\Users\\m.ahmadi\\Desktop\\FinalProject', 'C:\\Users\\mo.ahmadi\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\mo.ahmadi\\AppData\\Local\\Programs\\Python\\Python313\\DLLs']


In [2]:
print(sys.path)

['c:\\Users\\m.ahmadi\\Desktop\\FinalProject', 'C:\\Users\\mo.ahmadi\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\mo.ahmadi\\AppData\\Local\\Programs\\Python\\Python313\\DLLs', 'C:\\Users\\mo.ahmadi\\AppData\\Local\\Programs\\Python\\Python313\\Lib', 'C:\\Users\\mo.ahmadi\\AppData\\Local\\Programs\\Python\\Python313', 'c:\\Users\\m.ahmadi\\Desktop\\FinalProject\\venv', '', 'c:\\Users\\m.ahmadi\\Desktop\\FinalProject\\venv\\Lib\\site-packages']


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from env.loan_env import LoanEnv

### Import Dataset & Build env

In [4]:
PATH = r"C:\Users\m.ahmadi\Desktop\FinalProject\notebooks\df_cleaned.csv"

df_raw = pd.read_csv(PATH)

train_idx, test_idx = train_test_split(
    np.arange(len(df_raw)),
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = df_raw.iloc[train_idx].reset_index(drop=True)
test_df = df_raw.iloc[test_idx].reset_index(drop=True)

print("Train:", len(train_df))
print("Test :", len(test_df))

Train: 80000
Test : 20001


In [5]:
FEATURE_COLUMNS = [
    "MountlyIncome",
    "IncomeScore",
    "JobScore",
    "AssetScore",
    "TotalCustomerScore",
    "TotalFacilities",
    "DebtAmount",
    "OriginAmount",
    "LoanCounts",
    "DebtRatio",
    "DebtPerLoan",
    "Has_No_History",
    "Employed",
    "Score_Min",
    "Score_Max",
    "Risk_Min",
    "Risk_Max",
    "Risk",
]

train_features = train_df.copy()

train_features["Risk"] = (
    train_features["Risk_Min"] +
    train_features["Risk_Max"]
) / 2

scaler = StandardScaler()

scaler.fit(
    train_features[FEATURE_COLUMNS]
)

StandardScaler()

In [6]:
env = LoanEnv(test_df, scaler)

# Transition Model Behaviour Analysis

In [7]:
actions = [
    0,
    100_000_000,
    200_000_000,
    300_000_000,
    400_000_000,
    500_000_000,
]

rows = []

for i in range(len(test_df)):

    customer = env._row_to_customer_state(
        test_df.iloc[i]
    )

    risk_before = customer.risk

    for amount in actions:

        outcome = env.simulator.simulate(
            customer=customer.copy(),
            approved_unsecured_amount=amount
        )

        rows.append({
            "customer": i,
            "loan": amount,

            "risk_before": risk_before,
            "risk_after": outcome.next_state.risk,

            "delta_risk":
                outcome.next_state.risk - risk_before,

            "pd_before":
                env.simulator._calculate_default_probability(
                    env.simulator._estimate_hazard(customer)
                ),

            "pd_after":
                outcome.probability_of_default,

            "debt_ratio_change": (
                outcome.next_state.debt_ratio
                - customer.debt_ratio
                ),

            "debt_per_loan_change": (
                outcome.next_state.debt_per_loan
                - customer.debt_per_loan
                ),
        })

transition_df = pd.DataFrame(rows)

transition_df.shape

(120006, 9)

In [8]:
transition_df.groupby("loan")[
    ["risk_after", "delta_risk", "pd_after"]
].agg([
    "mean",
    "median",
    "std",
    "min",
    "max"
])

risk_after                                 delta_risk            \
                mean median        std   min     max       mean    median   
loan                                                                        
0          28.033936  20.65  25.297357  1.55   96.45   0.000000  0.000000   
100000000  28.828899  22.65  25.785406  0.00  100.00   0.794963  0.057326   
200000000  29.023238  22.65  25.826629  0.00  100.00   0.989302  0.183214   
300000000  29.205348  22.65  25.881499  0.00  100.00   1.171412  0.289274   
400000000  29.374790  22.65  25.936108  0.00  100.00   1.340854  0.392578   
500000000  29.533575  22.65  25.985838  0.00  100.00   1.499639  0.483679   

                                   pd_after                                \
                std    min    max      mean    median       std       min   
loan                                                                        
0          0.000000   0.00   0.00  0.014521  0.003698  0.022756  0.000830   
100000000  4.927489 -96.45  98.45  0.015121  0.004308  0.023236  0.000734   
200000000  5.130565 -96.45  98.45  0.015240  0.004308  0.023321  0.000734   
300000000  5.427234 -96.45  98.45  0.015356  0.004308  0.023417  0.000734   
400000000  5.717757 -96.45  98.45  0.015463  0.004308  0.023506  0.000734   
500000000  5.980704 -96.45  98.45  0.015564  0.004308  0.023587  0.000734   

                     
                max  
loan                 
0          0.081834  
100000000  0.082842  
200000000  0.082842  
300000000  0.082842  
400000000  0.082842  
500000000  0.082842

In [9]:
clipping_rate = (
    transition_df
    .groupby("loan")["risk_after"]
    .apply(lambda x: ((x == 0) | (x == 100)).mean())
)

clipping_rate

loan
0            0.00000
100000000    0.00580
200000000    0.00655
300000000    0.00730
400000000    0.00840
500000000    0.00925
Name: risk_after, dtype: float64

In [10]:
extreme_500 = transition_df[
    (transition_df["loan"] == 500_000_000) &
    (
        (transition_df["risk_after"] == 0) |
        (transition_df["risk_after"] == 100)
    )
]

extreme_500[
    [
        "customer",
        "risk_before",
        "risk_after",
        "delta_risk",
        "pd_before",
        "pd_after"
    ]
].head(20)

,customer,risk_before,risk_after,delta_risk,pd_before,pd_after
1919,319,61.75,100.0,38.25,0.047004,0.082842
2027,337,29.10,100.0,70.90,0.006982,0.082842
2591,431,20.65,100.0,79.35,0.003698,0.082842
3041,506,96.45,100.0,3.55,0.081834,0.082842
3767,627,96.45,100.0,3.55,0.081834,0.082842
4049,674,14.10,100.0,85.90,0.002229,0.082842
4601,766,9.40,100.0,90.60,0.001543,0.082842
5915,985,96.45,100.0,3.55,0.081834,0.082842
6017,1002,39.30,100.0,60.70,0.014323,0.082842
6239,1039,91.00,100.0,9.00,0.079691,0.082842


In [11]:
transition_df[
    (transition_df["loan"] == 500_000_000) &
    (transition_df["risk_after"] == 100)
][[
    "risk_before",
    "risk_after",
    "delta_risk",
    "debt_ratio_change",
    "debt_per_loan_change"
]].head(20)

,risk_before,risk_after,delta_risk,debt_ratio_change,debt_per_loan_change
1919,61.75,100.0,38.25,0.531228,2.499496e+08
2027,29.10,100.0,70.90,0.721056,2.486774e+08
2591,20.65,100.0,79.35,0.980392,2.500000e+08
3041,96.45,100.0,3.55,0.602764,2.446633e+08
3767,96.45,100.0,3.55,0.249999,2.499966e+08
4049,14.10,100.0,85.90,0.850340,2.500000e+08
4601,9.40,100.0,90.60,0.862403,2.491200e+08
5915,96.45,100.0,3.55,0.414238,1.526644e+08
6017,39.30,100.0,60.70,0.909091,2.500000e+08
6239,91.00,100.0,9.00,0.554219,2.500000e+08


In [12]:
customer = test_df.iloc[1919]

customer_state = env._row_to_customer_state(customer)

print("Debt:", customer_state.debt)
print("Loan count:", customer_state.loan_count)
print("Debt per loan:", customer_state.debt_per_loan)
print("Debt ratio:", customer_state.debt_ratio)

Debt: 3895973292.0
Loan count: 2
Debt per loan: 1947986646.0
Debt ratio: 1.207986776487805


In [13]:
amount = 500_000_000

outcome = env.simulator.simulate(
    customer_state.copy(),
    approved_unsecured_amount=amount
)

next_customer = outcome.next_state

print("BEFORE")
print("debt:", customer_state.debt)
print("loan_count:", customer_state.loan_count)
print("debt_per_loan:", customer_state.debt_per_loan)
print("debt_ratio:", customer_state.debt_ratio)

print("\nAFTER")
print("debt:", next_customer.debt)
print("loan_count:", next_customer.loan_count)
print("debt_per_loan:", next_customer.debt_per_loan)
print("debt_ratio:", next_customer.debt_ratio)

BEFORE
debt: 3895973292.0
loan_count: 2
debt_per_loan: 1947986646.0
debt_ratio: 1.207986776487805

AFTER
debt: 4395973292.0
loan_count: 3
debt_per_loan: 1465324430.6666667
debt_ratio: 1.2383023357746479


In [14]:
debt_per_loan_change = (
    next_customer.debt_per_loan -
    customer_state.debt_per_loan
) / customer_state.debt_per_loan

debt_ratio_change = (
    next_customer.debt_ratio -
    customer_state.debt_ratio
)

print("debt_per_loan_change:", debt_per_loan_change)
print("debt_ratio_change:", debt_ratio_change)

debt_per_loan_change: -0.24777490971225746
debt_ratio_change: 0.03031555928684293


In [16]:
debt_per_loan_change = (
    next_customer.debt_per_loan -
    customer_state.debt_per_loan
) / customer_state.debt_per_loan

debt_ratio_change = (
    next_customer.debt_ratio -
    customer_state.debt_ratio
)

delta_risk = (
    2 * debt_ratio_change
    + 1 * debt_per_loan_change
)

risk_manual = np.clip(
    customer_state.risk + delta_risk,
    0,
    100
)

print("debt_per_loan_change:", debt_per_loan_change)
print("debt_ratio_change:", debt_ratio_change)
print("delta_risk:", delta_risk)
print("risk_before:", customer_state.risk)
print("risk_manual:", risk_manual)
print("risk_simulator:", next_customer.risk)

debt_per_loan_change: -0.24777490971225746
debt_ratio_change: 0.03031555928684293
delta_risk: -0.1871437911385716
risk_before: 39.3
risk_manual: 39.11285620886142
risk_simulator: 39.11285620886142


# Loan Allocation Policy 